# 18. LangSmith 트레이싱 — 에이전트 관측 가능성
> Day 4 · 21H · 소요 약 50분

## 학습 목표

- LangSmith의 역할(LLM 애플리케이션 관측 가능성)을 설명할 수 있다.
- Run/Trace 구조를 이해하고 에이전트 실행을 자동으로 트레이싱할 수 있다.
- 토큰 사용량 / 지연 / 비용을 LangSmith API로 분석한다.
- 평가용 Dataset을 프로그래밍 방식으로 생성하고 Feedback을 부여한다.

> **전제 노트북:** 17번(`17_my_sql_agent.ipynb`)의 SQL 에이전트를 이 노트북에서 다시 이식합니다. 로직은 완전히 동일하며, LangSmith 트레이싱을 활성화해 각 노드·재시도·토큰·비용을 가시화합니다.
> **필요 키:** `OPENAI_API_KEY`, `NEON_DSN`, 그리고 **`LANGSMITH_API_KEY`** (LangSmith 무료 계정, https://smith.langchain.com/).

In [ ]:
%pip install -q langgraph langchain langchain-openai langsmith sqlalchemy psycopg2-binary sqlparse pandas matplotlib tabulate

In [ ]:
# Colab/로컬 환경에서 필요한 환경변수를 안전하게 로딩합니다 (다른 노트북과 동일 패턴).
import os


def _load_secret(key: str, required: bool = True) -> None:
    """Colab Secrets → getpass 입력 순으로 시도해 환경변수에 적재."""
    if os.environ.get(key):
        return
    value = None
    try:
        from google.colab import userdata  # type: ignore
        value = userdata.get(key)
    except Exception:
        value = None
    if not value:
        try:
            from getpass import getpass
            value = getpass(f"Enter {key}: ")
        except Exception:
            value = None
    if value:
        os.environ[key] = value
    elif required:
        raise RuntimeError(f"{key} is not set. Register it in Colab Secrets or via env var.")


# 이 노트북은 OpenAI(LLM) + Neon(DB) + LangSmith(트레이싱) 세 키가 모두 필요합니다.
_load_secret("OPENAI_API_KEY", required=True)
_load_secret("NEON_DSN", required=True)
_load_secret("LANGSMITH_API_KEY", required=True)

# ── LangSmith 자동 트레이싱 활성화 ─────────────────────────────────────
# 핵심 아이디어: 코드에 "트레이싱 코드" 를 넣지 않습니다. 대신 환경변수 몇 개만 설정해 두면
# LangChain/LangGraph 가 모든 호출을 자동으로 LangSmith 서버에 전송해 줍니다.
os.environ["LANGCHAIN_TRACING_V2"] = "true"        # 트레이싱 ON
os.environ["LANGCHAIN_PROJECT"] = os.environ.get("LANGCHAIN_PROJECT", "sql-agent-day4")  # 그룹 이름

# langsmith SDK 0.1.70 부터 환경변수 이름이 LANGSMITH_* 로 바뀌고 있습니다.
# 두 이름을 모두 세팅해 둬야 신구 버전 모두 호환됩니다 (어느 한쪽만 읽는 코드가 있어도 동작).
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = os.environ["LANGCHAIN_PROJECT"]

print("Environment ready.")
print(f"  LANGCHAIN_TRACING_V2 = {os.environ.get('LANGCHAIN_TRACING_V2')}")
print(f"  LANGCHAIN_PROJECT    = {os.environ.get('LANGCHAIN_PROJECT')}")
print(f"  LANGSMITH_API_KEY    = {'설정됨' if os.environ.get('LANGSMITH_API_KEY') else '미설정'}")

## 1. LangSmith란 무엇인가

전통 소프트웨어와 달리 LLM 애플리케이션은 **비결정적**입니다. 같은 질문에도 SQL 이 매번 다르게 생성될 수 있고, 실패 원인도 프롬프트 · 컨텍스트 · 모델 파라미터가 얽혀서 발생합니다. **관측 가능성(Observability)** 없이는 디버깅이 사실상 불가능합니다.

| 구분 | 전통 소프트웨어 | LLM 애플리케이션 |
|---|---|---|
| 출력 | 결정적 | 비결정적 |
| 버그 재현 | 쉬움 | 어려움 |
| 디버깅 도구 | 로그 / 디버거 | **트레이스 + 토큰 + 프롬프트 전문** |
| 평가 방법 | 단위 테스트 | 정량 평가 프레임워크 (Ragas 등) |

### Run / Trace 구조

```
Trace (에이전트 실행 1건)
├─ Run: generate_sql     (LLM 호출)
│    └─ input / output / tokens / latency / cost
├─ Run: execute_sql      (도구 실행)
├─ Run: validate_sql     (함수 실행)
└─ Run: generate_answer  (LLM 호출)
```

LangSmith 웹 UI (https://smith.langchain.com/) 에서는 **Trace 목록 → 개별 Trace 상세 → Run Tree → Analytics(통계)** 4단 메뉴로 이 데이터를 탐색합니다. 우리는 UI에서 눈으로 확인하는 것과 동시에, 이 노트북에서는 **`langsmith.Client` API**로 프로그램적으로 뽑아 분석합니다.

In [ ]:
# `langsmith.Client` 는 LangSmith 서버와 통신하는 SDK 클라이언트 객체입니다.
# 위 셀에서 환경변수를 정확히 설정해 뒀다면 인자 없이 Client() 만 해도 자동 인증됩니다.
from langsmith import Client

ls_client = Client()

# 연결 확인용 smoke test — 프로젝트 1 개만 listing 해 본다.
# 실패하면 (1) API 키 오타, (2) 네트워크 차단, (3) 무료 플랜 일일 한도 초과 등을 의심합니다.
try:
    _ = list(ls_client.list_projects(limit=1))
    print(f"LangSmith 연결 성공. 현재 프로젝트: {os.environ['LANGCHAIN_PROJECT']}")
except Exception as e:
    print(f"[WARN] LangSmith 접속 실패: {e}")

## 2. 17번 에이전트를 재이식

아래 셀들에서 17번 노트북의 에이전트를 **self-contained** 로 다시 세웁니다. 로직(상태·가드레일·4노드·재시도 분기)은 17번과 1:1 동일합니다. 차이는 환경변수로 **LangSmith 자동 트레이싱**이 켜져 있다는 것 뿐 — 별도 계측 코드는 필요 없습니다.

In [ ]:
import re
from typing import TypedDict, Optional, List, Dict, Any

from sqlalchemy import create_engine, inspect, text
import pandas as pd

from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# --- DB 엔진 (읽기 전용 모드 시도) ---
try:
    engine = create_engine(
        os.environ["NEON_DSN"],
        connect_args={"options": "-c default_transaction_read_only=on"},
        pool_pre_ping=True,
    )
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected in read-only mode.")
except Exception as e:
    print(f"[WARN] read-only 옵션 실패 → 일반 모드로 재연결: {e}")
    engine = create_engine(os.environ["NEON_DSN"], pool_pre_ping=True)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Engine connected (non-RO).")


In [ ]:
def collect_schema(engine, tables=None) -> str:
    """DB 스키마를 LLM 프롬프트용 DDL 텍스트로 변환 (NB17과 동일 로직)."""
    inspector = inspect(engine)
    if tables is None:
        tables = inspector.get_table_names()

    parts = []
    for table in tables:
        columns = inspector.get_columns(table)
        fks = inspector.get_foreign_keys(table)

        col_lines = []
        for col in columns:
            nullable = "" if col["nullable"] else " NOT NULL"
            col_lines.append(f"    {col['name']} {col['type']}{nullable}")

        fk_lines = []
        for fk in fks:
            fk_lines.append(
                f"    FOREIGN KEY ({', '.join(fk['constrained_columns'])}) "
                f"REFERENCES {fk['referred_table']}({', '.join(fk['referred_columns'])})"
            )

        ddl = f"CREATE TABLE {table} (\n"
        ddl += ",\n".join(col_lines)
        if fk_lines:
            ddl += ",\n" + ",\n".join(fk_lines)
        ddl += "\n);"

        try:
            with engine.connect() as conn:
                comments = conn.execute(
                    text("""
                        SELECT a.attname,
                               col_description(c.oid, a.attnum) AS comment
                        FROM pg_class c
                        JOIN pg_namespace n ON n.oid = c.relnamespace
                        JOIN pg_attribute a ON a.attrelid = c.oid
                        WHERE c.relname = :table
                          AND n.nspname = 'public'
                          AND a.attnum > 0
                          AND NOT a.attisdropped
                        ORDER BY a.attnum
                    """),
                    {"table": table},
                ).fetchall()
            for col_name, comment in comments:
                if comment:
                    ddl += f"\n-- {table}.{col_name}: {comment}"
        except Exception:
            pass

        parts.append(ddl)
    return "\n\n".join(parts)


TABLES = ["patients", "doctors", "visits", "diagnoses", "departments"]
_available = set(inspect(engine).get_table_names())
TABLES = [t for t in TABLES if t in _available]
if not TABLES:
    TABLES = list(_available)[:10]
print(f"Tables to include in schema: {TABLES}")

SCHEMA = collect_schema(engine, TABLES)
print(f"\nSchema text length: {len(SCHEMA)} chars")


In [ ]:
class AgentState(TypedDict, total=False):
    question: str
    sql: str
    result: List[Dict[str, Any]]
    result_md: str
    answer: str
    error: Optional[str]
    retry_count: int


BLOCKED_SQL_PATTERN = re.compile(
    r"\b(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|GRANT|REVOKE)\b",
    re.IGNORECASE,
)


def is_safe_sql(sql: str) -> tuple[bool, str]:
    match = BLOCKED_SQL_PATTERN.search(sql)
    if match:
        return False, f"보안 위반: '{match.group()}' 명령은 허용되지 않습니다."
    return True, ""


def inject_limit(sql: str, cap: int = 1000) -> str:
    stripped = sql.strip().rstrip(";")
    if re.search(r"\bLIMIT\s+\d+\b", stripped, re.IGNORECASE):
        return stripped
    return f"{stripped}\nLIMIT {cap}"


def strip_sql_fences(sql: str) -> str:
    sql = re.sub(r"```sql\s*", "", sql, flags=re.IGNORECASE)
    sql = re.sub(r"```\s*", "", sql)
    return sql.strip()


# 자가 테스트
for s in ["SELECT * FROM patients", "DROP TABLE visits"]:
    ok, reason = is_safe_sql(s)
    print(f"  is_safe_sql: ok={ok} reason='{reason}' | {s[:50]}")
print(inject_limit("SELECT * FROM patients"))

In [ ]:
sql_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
answer_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


SQL_GEN_TEMPLATE = ChatPromptTemplate.from_template(
    """당신은 PostgreSQL 전문가입니다. 아래 스키마를 참고하여 질문에 대한 SQL 하나를 작성하세요.

## 스키마
{schema}

## 규칙
- SELECT 문만 작성. DML/DDL 금지.
- visits.status = 'completed' 만 유효한 진료로 간주.
- 나이 = EXTRACT(YEAR FROM AGE(birth_date))
- 결과가 많을 가능성이 있으면 LIMIT 100 이하를 권장.
- 설명 없이 **SQL 만** 반환.
{error_feedback}

## 질문
{question}

SQL:
"""
)

sql_gen_chain = SQL_GEN_TEMPLATE | sql_llm | StrOutputParser()


def generate_sql(state: AgentState) -> dict:
    """노드 1 — 질문 → SQL. 이전 에러가 있으면 피드백으로 재생성."""
    error_feedback = ""
    if state.get("error"):
        error_feedback = (
            "\n## 직전 시도의 실패\n"
            f"- 실패 SQL:\n{state.get('sql', '')}\n"
            f"- 에러: {state['error']}\n"
            "이 에러를 피해서 SQL 을 다시 작성하세요."
        )

    raw = sql_gen_chain.invoke({
        "schema": SCHEMA,
        "question": state["question"],
        "error_feedback": error_feedback,
    })
    sql = strip_sql_fences(raw)
    return {
        "sql": sql,
        "retry_count": state.get("retry_count", 0) + 1,
    }


def execute_sql(state: AgentState) -> dict:
    """노드 2 — SQL 실행 + 결과 Markdown 화."""
    sql = state.get("sql", "")
    if not sql:
        return {"error": "실행할 SQL 이 없습니다.", "result": [], "result_md": ""}

    ok, reason = is_safe_sql(sql)
    if not ok:
        return {"error": reason, "result": [], "result_md": ""}

    safe_sql = inject_limit(sql, cap=1000)
    try:
        df = pd.read_sql(text(safe_sql), engine)
        if df.empty:
            return {
                "result": [],
                "result_md": "(결과 없음 — 조건을 다시 확인하세요)",
                "error": "",
                "sql": safe_sql,
            }
        result_rows = df.head(50).to_dict(orient="records")
        md_table = df.head(50).to_markdown(index=False)
        if len(df) > 50:
            md_table += f"\n\n... 외 {len(df) - 50}행"
        return {
            "result": result_rows,
            "result_md": md_table,
            "error": "",
            "sql": safe_sql,
        }
    except Exception as e:
        return {
            "error": f"SQL 실행 오류: {type(e).__name__}: {str(e)[:300]}",
            "result": [],
            "result_md": "",
        }


def validate_sql(state: AgentState) -> dict:
    """노드 3 — 단순 검증. 에러가 있으면 그대로 두고 분기 함수가 retry 결정."""
    if state.get("error"):
        return {}
    return {"error": ""}


ANSWER_TEMPLATE = ChatPromptTemplate.from_template(
    """다음 SQL 실행 결과를 바탕으로 질문에 한국어로 답변하세요.

## 질문
{question}

## 실행한 SQL
{sql}

## 결과 (Markdown 표)
{result_md}

## 규칙
- 2-3 문장으로 핵심만 요약.
- 숫자에 천 단위 구분자(쉼표) 사용.
- 결과가 비어 있으면 "해당 조건에 맞는 데이터가 없습니다." 로 시작하는 안내.
- 추측 금지 — 표에 없는 수치는 언급하지 말 것.
"""
)

answer_chain = ANSWER_TEMPLATE | answer_llm | StrOutputParser()


def generate_answer(state: AgentState) -> dict:
    """노드 4 — 결과 → 자연어 답변."""
    if state.get("error") and state.get("retry_count", 0) >= 3:
        return {
            "answer": (
                "죄송합니다. 질문에 답변하지 못했습니다.\n"
                f"- 마지막 에러: {state['error']}\n"
                "- 질문을 더 구체적으로 다시 물어봐 주세요."
            )
        }
    ans = answer_chain.invoke({
        "question": state.get("question", ""),
        "sql": state.get("sql", ""),
        "result_md": (state.get("result_md") or "(결과 없음)")[:1500],
    })
    return {"answer": ans.strip()}


MAX_RETRIES = 3


def should_retry(state: AgentState) -> str:
    if not state.get("error"):
        return "answer"
    if state.get("retry_count", 0) >= MAX_RETRIES:
        return "giveup"
    return "retry"

print("Nodes defined:", ["generate_sql", "execute_sql", "validate_sql", "generate_answer"])

In [ ]:
graph = StateGraph(AgentState)
graph.add_node("generate_sql", generate_sql)
graph.add_node("execute_sql", execute_sql)
graph.add_node("validate_sql", validate_sql)
graph.add_node("generate_answer", generate_answer)

graph.set_entry_point("generate_sql")
graph.add_edge("generate_sql", "execute_sql")
graph.add_edge("execute_sql", "validate_sql")
graph.add_conditional_edges(
    "validate_sql",
    should_retry,
    {
        "answer": "generate_answer",
        "giveup": "generate_answer",
        "retry":  "generate_sql",
    },
)
graph.add_edge("generate_answer", END)

agent = graph.compile()
print("Agent compiled.")

def _initial_state(question: str) -> dict:
    return {
        "question": question,
        "sql": "",
        "result": [],
        "result_md": "",
        "answer": "",
        "error": "",
        "retry_count": 0,
    }


## 3. 10개 질문 일괄 실행 — 자동 트레이싱

`LANGCHAIN_TRACING_V2="true"` 가 켜져 있으면, 모든 LangChain / LangGraph 실행이 **자동으로** LangSmith에 기록됩니다. 코드에 아무 계측도 추가할 필요가 없습니다.

In [ ]:
# 10개 질문을 일괄 실행. 위 셀에서 LangSmith 트레이싱이 켜져 있으므로,
# 이 안에서 일어나는 모든 LLM/툴 호출이 자동으로 LangSmith 서버에 기록됩니다 — 별도 코드 불필요.
project_questions = [
    "전체 환자 수는?",
    "남성 환자 중 40세 이상은 몇 명?",
    "진료과별 의사 수를 보여줘",
    "지난달 완료 진료 건수는?",
    "응급 진료 평균 비용은?",
    "가장 많이 방문한 환자 Top 3는?",
    "중증 진단을 받은 환자 이름은?",
    "2026년 월별 방문 수 추이는?",
    "내과 의사 중 급여 최고는?",
    "혈액형별 환자 분포는?",
]

results = []
# enumerate(..., start=1) 처럼 시작값을 줄 수도 있지만, 여기는 i+1 로 1-based 출력.
for i, q in enumerate(project_questions):
    # agent.invoke() 는 그래프 끝까지 실행해 최종 state 를 돌려준다.
    # (.stream() 은 단계별 이벤트, .invoke() 는 최종 결과만 반환)
    state = agent.invoke(_initial_state(q))
    # status 판정: answer 가 비어 있지 않고 error 가 비어 있어야 OK.
    ok = bool(state.get("answer")) and not state.get("error")
    status = "OK" if ok else "FAIL"
    results.append({
        "question": q,
        "status": status,
        "retries": state.get("retry_count", 0),
        # 화면에 너무 길게 찍히지 않도록 줄바꿈 제거 + 80자에서 자르기
        "sql": (state.get("sql") or "").replace("\n", " ")[:80],
        "answer": (state.get("answer") or "").replace("\n", " ")[:80],
    })
    # f"{i+1:2d}" → 정수를 2자리 폭으로 출력 (예: " 1", "10").
    print(f"[{i+1:2d}/{len(project_questions)}] {status} (retries={state.get('retry_count', 0)}) — {q}")

# 결과를 DataFrame 으로 만들면 셀 마지막 줄에 표가 바로 렌더링됩니다.
df_results = pd.DataFrame(results)
success = (df_results["status"] == "OK").sum()
print(f"\n정답률: {success}/{len(project_questions)} ({success/len(project_questions)*100:.0f}%)")
print(f"LangSmith UI: https://smith.langchain.com/  (project: {os.environ['LANGCHAIN_PROJECT']})")
df_results

## 4. LangSmith API로 트레이스 분석

LangSmith UI에서 눈으로 보는 것 뿐만 아니라, `list_runs` 로 프로그램적으로 모든 실행 데이터를 끌어올 수 있습니다. 여기서는 질문별 토큰 / 지연 / 예상 비용을 DataFrame으로 만들어 시각화합니다.

> **주의:** LangSmith 서버에 실행 데이터가 도착하기까지 보통 수 초가 걸립니다. 아래 셀에서 `list_runs` 결과가 비어 있으면 15~30초 대기 후 재실행하세요.

In [ ]:
# 위에서 일괄 실행한 트레이스를 LangSmith 서버에서 다시 끌어옵니다.
# 토큰 / 지연 / 비용 통계를 직접 분석하기 위해서입니다.
import time

# 트레이스가 서버 DB 에 반영될 시간을 잠시 기다립니다 (네트워크/배치 지연 대비).
time.sleep(5)

# is_root=True : 그래프 1회 실행 단위(root run) 만 가져온다.
# 자식 노드 단계까지 모두 가져오면 양이 너무 많아 분석이 어려워집니다.
#
# 구버전 langsmith SDK 는 is_root 인자를 모르고 execution_order 만 받습니다.
# TypeError 가 나면 자동으로 폴백해서 양쪽 SDK 모두 동작하도록 합니다.
try:
    runs = list(ls_client.list_runs(
        project_name=os.environ["LANGCHAIN_PROJECT"],
        is_root=True,
        limit=len(project_questions) * 2,  # 약간 여유 있게
    ))
except TypeError:
    runs = list(ls_client.list_runs(
        project_name=os.environ["LANGCHAIN_PROJECT"],
        execution_order=1,                  # 구버전: 1 = root
        limit=len(project_questions) * 2,
    ))

print(f"LangSmith 에서 {len(runs)}개 root run 수집")

In [ ]:
# run 객체에서 토큰/지연/비용을 뽑아 DataFrame 으로 정리합니다.
# 토큰 필드 위치는 SDK 버전에 따라 달라지므로 "여러 위치를 시도하는" 도우미 함수를 정의합니다.
def _token_usage(run):
    """run에서 토큰 수를 방어적으로 추출 (SDK 버전마다 필드가 다름)."""
    # 1순위: 최상위 속성 (최신 SDK)
    total = getattr(run, "total_tokens", None)
    prompt = getattr(run, "prompt_tokens", None)
    completion = getattr(run, "completion_tokens", None)
    if not total:
        # 2순위: extra.runtime.token_usage 안에 들어 있는 경우 (구버전)
        extra = (getattr(run, "extra", None) or {}).get("runtime", {}).get("token_usage", {}) or {}
        prompt = prompt or extra.get("prompt_tokens", 0)
        completion = completion or extra.get("completion_tokens", 0)
        total = extra.get("total_tokens", (prompt or 0) + (completion or 0))
    # `or 0` → None 이면 0으로 안전 변환.
    return int(total or 0), int(prompt or 0), int(completion or 0)


trace_stats = []
for run in runs:
    total, prompt_t, completion_t = _token_usage(run)
    # 지연(latency) = 끝난 시각 - 시작한 시각. 둘 중 하나라도 None 이면 0 으로 처리.
    if run.end_time and run.start_time:
        latency = (run.end_time - run.start_time).total_seconds()
    else:
        latency = 0.0
    # 가격 계산: gpt-4o-mini 의 2026 기준 단가 ($/1M tokens) — 입력 0.15, 출력 0.60.
    # (prompt_t 와 completion_t 단위는 토큰 1개)
    cost = (prompt_t * 0.15 + completion_t * 0.60) / 1_000_000
    question = (run.inputs or {}).get("question", "?")
    trace_stats.append({
        "question": question[:30],   # 표 폭을 위해 30자에서 자름
        "tokens":   total,
        "latency_s": round(latency, 2),
        "cost_usd": round(cost, 6),  # 비용은 6자리까지 — 한 건이 1¢ 미만이라 자릿수 필요
        "status":   getattr(run, "status", "?"),
    })

df_traces = pd.DataFrame(trace_stats)
print(df_traces.to_string(index=False))

# 비어 있지 않을 때만 합계 출력 (LangSmith 반영 지연으로 비어 있을 수 있음).
if not df_traces.empty:
    print(f"\n총 토큰:   {df_traces['tokens'].sum():,}")
    print(f"평균 지연: {df_traces['latency_s'].mean():.2f} s")
    print(f"총 비용:   ${df_traces['cost_usd'].sum():.4f}")

## 5. 시각화 — 질문별 토큰/지연 2-패널 차트

막대 길이로 어떤 질문이 토큰을 많이 쓰고 어떤 질문이 느린지를 한눈에 파악합니다. 재시도가 발생한 질문은 보통 토큰·지연이 모두 큰 봉우리로 나타납니다.

In [ ]:
import matplotlib.pyplot as plt

if df_traces.empty:
    print("[skip] df_traces가 비어 있어 시각화를 건너뜁니다. 30초 뒤 위 셀을 재실행하세요.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].barh(range(len(df_traces)), df_traces["tokens"], color="steelblue")
    axes[0].set_yticks(range(len(df_traces)))
    axes[0].set_yticklabels(df_traces["question"], fontsize=8)
    axes[0].set_xlabel("Total Tokens")
    axes[0].set_title("Tokens per Question")
    axes[0].invert_yaxis()

    axes[1].barh(range(len(df_traces)), df_traces["latency_s"], color="coral")
    axes[1].set_yticks(range(len(df_traces)))
    axes[1].set_yticklabels(df_traces["question"], fontsize=8)
    axes[1].set_xlabel("Latency (seconds)")
    axes[1].set_title("Latency per Question")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.savefig("langsmith_stats.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("차트 저장: langsmith_stats.png")

### LangSmith UI 연계 실습

1. 위 차트에서 **지연이 가장 긴 질문**을 하나 고르세요.
2. https://smith.langchain.com/ 에 로그인 → 현재 프로젝트(`sql-agent-day4`) 를 엽니다.
3. 해당 질문의 Trace를 클릭 → **Run Tree** 화면을 엽니다.
4. `generate_sql` / `execute_sql` / `validate_sql` / `generate_answer` 노드의 소요 시간을 비교하고 병목 노드를 찾으세요.
5. 재시도가 있었다면 `generate_sql` 가 2번 이상 나타날 겁니다 — 첫 번째와 두 번째의 프롬프트 / 출력 차이를 비교하세요.

> **FAQ — "UI에 아무것도 안 보여요"**
>
> - `LANGSMITH_API_KEY` 를 다시 확인하세요 (만료 / 오타 가능).
> - 처음 트레이스가 서버에 반영되기까지 최대 1분 지연될 수 있습니다.
> - Colab 무료 계정은 네트워크가 가끔 끊깁니다 — `runs` 가 0개면 15~30초 뒤 위 셀만 재실행.

## 6. 평가용 Dataset 생성

LangSmith **Dataset** = "질문 + 기대 정답(ground truth)" 의 모음. 이 Dataset은 다음 노트북(`19_ragas_eval.ipynb`) 에서 Ragas 정량 평가의 ground_truth로 재사용됩니다.

아래 셀은:
1. 같은 이름의 데이터셋이 이미 있으면 **삭제 후 재생성** (멱등 실행).
2. 질문 10개 + 병원 DB 기준 간단 ground_truth 를 `create_example` 로 등록.

In [ ]:
# 평가용 Dataset 만들기 — "질문 + 기대 정답" 의 모음.
# 다음 노트북(19_ragas_eval) 에서 ground_truth 로 재사용하므로 의미 있는 출력 형식을 갖춰 둡니다.
dataset_name = "sql-agent-eval-10q"

# 같은 이름의 Dataset 이 이미 있으면 먼저 삭제 → 셀을 여러 번 실행해도 깨끗한 상태를 유지(idempotent).
try:
    existing = ls_client.read_dataset(dataset_name=dataset_name)
    ls_client.delete_dataset(dataset_id=existing.id)
    print(f"(overwrite) 기존 dataset '{dataset_name}' 삭제")
except Exception:
    # 처음 실행 시에는 read_dataset 이 NotFound 를 던질 수 있어 그냥 무시.
    pass

dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="SQL 에이전트 10개 질문 평가용 데이터셋 (병원 DB 기준)",
)

# 질문(input) 과 기대 정답(output) 페어. ground_truth 는 사람이 검토해 작성한 정답 기준선.
ground_truths = [
    {"question": "전체 환자 수는?",                      "ground_truth": "전체 환자 수는 30명입니다."},
    {"question": "남성 환자 중 40세 이상은 몇 명?",      "ground_truth": "남성 환자 중 40세 이상은 약 7명입니다."},
    {"question": "진료과별 의사 수를 보여줘",            "ground_truth": "진료과별 의사 수 — 내과 3명, 외과 3명, 소아과 3명, 정형외과 2명, 피부과 2명, 신경과 3명, 산부인과 2명, 안과 2명."},
    {"question": "지난달 완료 진료 건수는?",            "ground_truth": "지난달(이전 달) visits.status='completed' 인 건수를 반환."},
    {"question": "응급 진료 평균 비용은?",               "ground_truth": "응급 진료 평균 비용은 약 300,000원."},
    {"question": "가장 많이 방문한 환자 Top 3는?",       "ground_truth": "방문 횟수 내림차순 상위 3명의 환자 이름."},
    {"question": "중증 진단을 받은 환자 이름은?",        "ground_truth": "급성 충수염, 담낭결석, 뇌진탕 등 severity='severe' 진단 환자."},
    {"question": "2026년 월별 방문 수 추이는?",          "ground_truth": "2026년 1~4월 월별 방문 수."},
    {"question": "내과 의사 중 급여 최고는?",             "ground_truth": "내과 의사 중 김철수가 월 8,500,000원으로 최고."},
    {"question": "혈액형별 환자 분포는?",                "ground_truth": "A/B/O/AB 형별 환자 수 분포."},
]

# 신버전 SDK 는 한 번에 여러 개 등록(create_examples), 구버전은 1개씩 등록(create_example).
# 둘 다 시도해서 호환성을 확보.
try:
    ls_client.create_examples(
        inputs=[{"question": gt["question"]} for gt in ground_truths],
        outputs=[{"ground_truth": gt["ground_truth"]} for gt in ground_truths],
        dataset_id=dataset.id,
    )
except Exception:
    for gt in ground_truths:
        ls_client.create_example(
            inputs={"question": gt["question"]},
            outputs={"ground_truth": gt["ground_truth"]},
            dataset_id=dataset.id,
        )

print(f"Dataset '{dataset_name}' created with {len(ground_truths)} examples.")
print(f"  id = {dataset.id}")

## 7. Feedback 태깅 — 성공/실패 라벨 부여

**Feedback** = 개별 Run에 점수·코멘트를 붙이는 기능. 정답/오답을 태깅해 두면 LangSmith UI에서 "실패한 Run 만" 필터링해 디버깅할 수 있습니다.

> **주의:** `list_runs()` 는 기본적으로 **최신 → 과거** 역순입니다. 그냥 `zip(runs, results)` 하면 엉뚱한 Run에 Feedback이 붙습니다. 아래는 **질문 문자열로 매칭** 하는 안전한 방법.

In [ ]:
# Run 별로 Feedback 점수(0/1)를 부여 — UI 에서 실패 케이스만 필터링할 수 있게 됩니다.
# ⚠️ list_runs() 는 기본 정렬이 "최신 → 과거" 라 zip 으로 results 와 짝지으면 어긋납니다.
# 안전한 방법: run.inputs.question 텍스트를 키로 dict 매칭.

# 방금 실행한 run 들을 다시 가져온다.
try:
    recent = list(ls_client.list_runs(
        project_name=os.environ["LANGCHAIN_PROJECT"],
        is_root=True,
        limit=len(results),
    ))
except TypeError:
    # 구버전 SDK 폴백
    recent = list(ls_client.list_runs(
        project_name=os.environ["LANGCHAIN_PROJECT"],
        execution_order=1,
        limit=len(results),
    ))

# 같은 질문이 여러 번 실행됐으면 가장 최근 것만 보관 (start_time 오름차순으로 덮어쓰기).
runs_by_question: Dict[str, Any] = {}
for r in sorted(recent, key=lambda r: r.start_time):
    q = (r.inputs or {}).get("question")
    if q is not None:
        runs_by_question[q] = r

feedback_count = 0
for res in results:
    run = runs_by_question.get(res["question"])
    if run is None:
        # 매칭 실패 — 보통 SDK 가 텍스트를 자르거나 키 이름이 다른 경우.
        print(f"  (skip) run 매칭 실패: {res['question'][:30]}")
        continue
    # OK = 1.0, FAIL = 0.0 으로 점수화. UI 에서 정렬·필터 가능.
    score = 1.0 if res["status"] == "OK" else 0.0
    ls_client.create_feedback(
        run_id=run.id,
        key="correctness",  # UI 에서 보일 라벨
        score=score,
        comment=f"{res['question'][:40]} -> {res['status']} (retries={res['retries']})",
    )
    feedback_count += 1
    print(f"  feedback: {res['question'][:40]} score={score}")

print(f"\n{feedback_count}/{len(results)} run 에 Feedback(correctness) 태깅 완료.")
print("LangSmith UI → 'Feedback' 필터에서 score=0 으로 실패 케이스만 추릴 수 있습니다.")

## 실습 과제

1. **본인 프로젝트 에이전트에 LangSmith 트레이싱 연결**: `LANGCHAIN_PROJECT` 를 본인 프로젝트명(예: `sql-agent-{yourname}`)으로 바꾸고 본인 질문 10개를 실행하세요. 과제 #3 제출 시 트레이스 URL (`https://smith.langchain.com/...`) 을 함께 공유합니다.
2. **병목 분석**: 위에서 수집한 `df_traces` 에서 `latency_s` 가 가장 큰 질문 1건의 Trace를 UI에서 열어 어떤 노드가 병목인지 한 줄로 메모하세요.
3. **(도전) 본인 도메인 Dataset**: `ground_truths` 대신 본인 도메인의 질문 + 기대 정답 10개로 Dataset을 생성하세요. 다음 노트북(Ragas)에서 이 Dataset을 ground_truth 로 사용합니다.

In [ ]:
# ============================================================
# TODO — 본인 프로젝트에 LangSmith 연결
# ============================================================
# 1) 프로젝트 이름을 본인 이름으로 (LangSmith UI 에서 구분되도록)
# 여기에 구현하세요.
#
# 2) 본인 도메인 질문 10개
# 여기에 구현하세요.
#
# 3) 실행 — agent.invoke 호출만 해도 자동으로 LangSmith에 기록됨
# 여기에 구현하세요.
#
# 4) LangSmith UI 에서 trace URL 을 복사해 과제 #3 제출에 포함
# 여기에 구현하세요.

## 다음 노트북에서는...

`19_ragas_eval.ipynb` 에서는 이 에이전트가 "정확한 답을 내는지" 를 **정량 평가** 합니다. LangSmith가 "어떻게 실행되었는가"(지연·토큰·분기)를 보여줬다면, Ragas는 "얼마나 좋은 답이었는가"(Faithfulness / Answer Relevancy / Context Precision·Recall)를 4개 지표로 채점합니다. 프롬프트 튜닝 전/후 비교 차트를 만들어 최종 발표의 핵심 슬라이드로 활용합니다.